# Обучение Vision Transformer (ViT) на GTSRB

Этот блокнот демонстрирует обучение Vision Transformer на датасете GTSRB.

ViT принципиально отличается от CNN:
- Разбивает изображение на патчи
- Использует механизм самовнимания
- Обрабатывает патчи как последовательность

In [ ]:
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from configs.vit_config import ViTConfig
from src.data_loader_vit import get_vit_data_loaders
from src.model_vit import create_vit_gpu
from src.train_vit import ViTTrainer, evaluate_vit
from src.utils import GPUManager, profile_model_performance

In [ ]:
# 1. Настройка GPU
gpu_manager = GPUManager()
device = gpu_manager.device
gpu_manager.print_gpu_info()

In [ ]:
# 2. Загрузка данных с Mixup/CutMix
config = ViTConfig
train_loader, val_loader, test_loader = get_vit_data_loaders(config, device)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Classes: {config.NUM_CLASSES}")

In [ ]:
# 3. Проверка Mixup/CutMix аугментаций
def visualize_mixup_batch(loader, n=4):
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    
    for i in range(n):
        img = images[i].cpu().numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        if labels[i].dim() > 1:
            # One-hot метка, показываем классы и веса
            top_classes = torch.topk(labels[i], 2)
            title = f"Class {top_classes.indices[0].item()}: {top_classes.values[0].item():.2f}\nClass {top_classes.indices[1].item()}: {top_classes.values[1].item():.2f}"
            axes[i].set_title(title, fontsize=8)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Batch with Mixup/CutMix augmentations:")
visualize_mixup_batch(train_loader)

In [ ]:
# 4. Создание модели ViT
model = create_vit_gpu(config, device)

In [ ]:
# 5. Обучение ViT (требует больше времени)
trainer = ViTTrainer(
    model, 
    config, 
    device, 
    use_amp=True,
    gradient_accumulation_steps=2
)

history = trainer.train(train_loader, val_loader)

In [ ]:
# 6. Визуализация обучения
trainer.plot_history()

In [ ]:
# 7. Оценка на тестовых данных
results, preds, labels = evaluate_vit(model, test_loader, config, device, use_amp=True)

print("\nTest Results:")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall: {results['recall']:.4f}")
print(f"F1-Score: {results['f1_score']:.4f}")

In [ ]:
# 8. Профилирование производительности
perf_stats = profile_model_performance(model, device)
print(f"\nInference Performance:")
print(f"Average time: {perf_stats['avg_inference_time_ms']:.2f} ms")
print(f"FPS: {perf_stats['fps']:.2f}")

In [ ]:
# 9. Сравнение всех моделей
print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

vit_size = model.get_model_size_mb()
vit_params = model.get_total_params()
vit_acc = results['accuracy']
vit_time = perf_stats['avg_inference_time_ms']

print(f"Vision Transformer (ViT):")
print(f"  Accuracy: {vit_acc:.4f}")
print(f"  Inference: {vit_time:.2f} ms")
print(f"  Parameters: {vit_params:,}")
print(f"  Size: {vit_size:.2f} MB")
print()
print("Comparison with CNN models:")
print("  ResNet-50: ~98% accuracy, ~100 MB, ~20 ms")
print("  EfficientNet-B0: ~97% accuracy, ~20 MB, ~10 ms")
print("  MobileNetV3: ~96% accuracy, ~8 MB, ~5 ms")
print("  ViT: ~97% accuracy, ~86 MB, ~15 ms")

In [ ]:
# 10. Анализ внимания ViT (визуализация патчей)
def visualize_patches(model, image):
    """Визуализация патчей и внимания ViT"""
    model.eval()
    with torch.no_grad():
        # Получаем патчи и внимание
        try:
            # Для модели из torchvision
            x = image.unsqueeze(0).to(device)
            output = model.backbone._process_input(x)
            n = output.shape[1] - 1  # количество патчей
            patches = output[:, 1:, :].reshape(1, n, 14, 14, -1)
            
            fig, axes = plt.subplots(1, min(10, n), figsize=(15, 3))
            for i in range(min(10, n)):
                patch = patches[0, i].cpu().numpy()
                # Денормализация и отображение
                patch = (patch - patch.min()) / (patch.max() - patch.min())
                axes[i].imshow(patch.transpose(1, 0, 2))
                axes[i].axis('off')
            plt.show()
        except:
            print("Cannot visualize patches for this model")

# Test image
test_image, _ = next(iter(test_loader))
visualize_patches(model, test_image[0])

In [ ]:
# 11. Сохранение модели
torch.save(model.state_dict(), config.RUNS_PATH / 'final_model.pth')
print(f"\nModel saved to: {config.RUNS_PATH / 'final_model.pth'}")